# minipi05 Colab smoke test

Runtime: **GPU** (T4 is enough). Needs **write-less** public HF repos, or set `HF_TOKEN` for private models.

Use a recent commit that includes `minipi05/stm/` (RT) — install from your branch if needed.

In [ ]:
# Optional: Secrets → HF_TOKEN (read is enough for public model repos)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN secret (OK if repos are public)")

In [ ]:
# Install (Colab already has torch). Pin your fork/branch if github is behind:
# %pip install -q "minipi05[gpu] @ git+https://github.com/lucehe/minipi05.git@main"
%pip install -q "git+https://github.com/lucehe/minipi05.git"
%pip install -q triton  # Colab Linux: faster GDN/STM paths

In [ ]:
!python -c "import minipi05; print('version', minipi05.__version__)"

In [ ]:
# One RT LIBERO student from catalog (HF download → ~/.cache/minipi05)
!python -m minipi05.demo --list-experts 2>/dev/null | head -20 || true
!wget -q -O /tmp/colab_smoke.py https://raw.githubusercontent.com/lucehe/minipi05/main/scripts/colab_smoke.py 2>/dev/null || true

In [ ]:
import os, sys, torch
from pathlib import Path

repo = Path("/content/minipi05")
if repo.is_dir():
    sys.path.insert(0, str(repo))

from minipi05.experts import load_expert_policies

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable Runtime → GPU"

for expert_id in ["rt_libero_s0", "rt_libero_s1", "rt_libero_s2", "rt_libero_s3"]:
    print("===", expert_id, "===")
    pol = load_expert_policies([expert_id], device=device)[0]
    cfg = pol.config
    images = torch.rand(1, cfg.num_cameras, 3, cfg.image_size, cfg.image_size, device=device)
    state = torch.randn(1, cfg.state_dim, device=device)
    with torch.inference_mode():
        chunk = pol.predict_chunk(images, state)
    print("delta", cfg.delta_type, "chunk", tuple(chunk.shape), "mean", float(chunk.mean()))

In [ ]:
from minipi05 import ExpertEnsemble, load_expert_policies

policies = load_expert_policies(
    ["rt_libero_s0", "rt_libero_s1"], device=device
)
blend = ExpertEnsemble(policies)
with torch.inference_mode():
    chunk = blend.predict_chunk(images, state)
print("ensemble", tuple(chunk.shape))